# Round 3 on Kaggle (T4)

Same runs, configs and analysis as `Round3_Decisive_Colab.ipynb`; only the plumbing differs (Kaggle has no Google Drive, its inputs are read-only, and a session stops after 12 hours).

**Which parts run here.** Keep every comparison on one platform: **only C (VLCS) runs on Kaggle**. A, B and D run on Colab (pre-registration, Amendment 1). Every pair the analysis compares then comes from the same platform. Never run the same part on both.

| Part | Runs | T4 time |
|---|---|---|
| C. VLCS: M2, M2-CL, ERM, seeds 0-2 | 36 | about 15-18 h (2 sessions) |

**One-time setup**
1. Phone-verify your Kaggle account (Settings > Phone verification); without it there is no GPU or internet.
2. Create a private dataset (Datasets > New Dataset) named `m2cl-inputs` holding `M2CL_code.zip` (from the repo's `colab_upload/`) and `PACS.zip`. Kaggle unpacks the zips; this notebook finds them either way.
3. Notebooks > New Notebook > File > Import Notebook > this file. In the right panel: **Accelerator: GPU T4 x2** (not P100), **Internet: on**, and Add Input > Your Work > Datasets > `m2cl-inputs`.

**Each session**
1. Set `RUN_PARTS` in the first cell.
2. Click **Save Version > Save & Run All (Commit)**. It runs in the background (you can close the browser), stops starting new runs before the 12-hour limit, and saves `/kaggle/working/m2cl_v2` as the version's output.
3. **From the second session on**, first add the previous version's output as an input: Add Input > Your Work > Notebooks > this notebook (make sure it shows the newest version). The second cell copies that progress in, and unfinished runs resume from their last epoch.

Every version's output holds everything finished so far, so the newest one is all you need to download at the end.

In [ ]:
# 1. Setup. Choose the parts this session runs.
RUN_PARTS = ['C']          # Kaggle runs only C; A, B and D run on Colab (pre-registration, Amendment 1).

import glob, json, os, platform, shutil, subprocess, sys, time, urllib.request, zipfile
import torch, torchvision
START = time.time()
INPUT = '/kaggle/input'
OUT   = '/kaggle/working/m2cl_v2'   # saved as this version's output
WORK  = '/tmp'                      # code and data; not saved
assert set(RUN_PARTS) <= {'A', 'B', 'C', 'D'}, RUN_PARTS

assert torch.cuda.is_available(), 'No GPU: in the right panel choose Accelerator > GPU T4 x2, then run again.'
GPU = torch.cuda.get_device_name(0)
assert 'T4' in GPU, f'{GPU}: choose GPU T4 x2, not P100, so these runs use the same GPU type as the Colab runs.'
try:
    urllib.request.urlopen('https://download.pytorch.org/models/', timeout=15)
except urllib.error.HTTPError:
    pass                            # the server answered, so the internet works
except Exception as e:
    raise AssertionError('Internet is off. Turn it on in the right panel (needs a phone-verified account); '
                         'it is needed for the ImageNet weights and VLCS.') from e

# Code: the unpacked M2CL_code.zip (or the zip itself) from the inputs
def code_ok(root):
    s = os.path.join(root, 'src', 'm2cl', 'splits.py')
    return os.path.exists(s) and 'detect_domains' in open(s, encoding='utf-8').read() \
        and os.path.exists(os.path.join(root, 'configs', 'round3_m2cl.yaml'))
roots = [os.path.dirname(os.path.dirname(p)) for p in glob.glob(f'{INPUT}/**/scripts/train_lodo.py', recursive=True)]
roots = [r for r in roots if code_ok(r)]
if not roots:
    for z in glob.glob(f'{INPUT}/**/M2CL_code*.zip', recursive=True):
        shutil.rmtree(f'{WORK}/code_unzip', ignore_errors=True)
        zipfile.ZipFile(z).extractall(f'{WORK}/code_unzip')
        if code_ok(f'{WORK}/code_unzip'):
            roots = [f'{WORK}/code_unzip']
            break
assert roots, 'No round-3 M2CL_code found in the inputs. Add the m2cl-inputs dataset with the new M2CL_code.zip.'
shutil.rmtree(f'{WORK}/repo', ignore_errors=True)
shutil.copytree(roots[0], f'{WORK}/repo')
os.chdir(f'{WORK}/repo')
sys.path.insert(0, 'src')
print('Code:', roots[0])
# VLCS has truncated JPEGs; the loader decodes them as DomainBed does (docs/AUDIT.md, F17). A code zip built
# before 7 Nov 2026 lacks that line, so add exactly the line the current code has.
FLAG = 'ImageFile.LOAD_TRUNCATED_IMAGES = True'
data_py = open('src/m2cl/data.py', encoding='utf-8').read()
PATCHED = FLAG not in data_py
if PATCHED:
    NL = chr(10)
    data_py = data_py.replace('from PIL import Image' + NL, 'from PIL import Image, ImageFile' + NL, 1)
    data_py = data_py.replace(NL + 'MEAN, STD', NL + FLAG + NL + NL + 'MEAN, STD', 1)
    assert data_py.count(FLAG) == 1 and 'from PIL import Image, ImageFile' in data_py, \
        'Could not add the truncated-image line.'
    open('src/m2cl/data.py', 'w', encoding='utf-8').write(data_py)
    print('Added the truncated-image line to this older code zip.')

# PACS: the RGB copy (the folder of four domains whose files are not dct_*; docs/AUDIT.md, A0)
def is_rgb(root):
    folder = os.path.join(root, 'art_painting', 'dog')
    return os.path.isdir(folder) and not any(f.startswith('dct_') for f in os.listdir(folder))
def find_pacs(base):
    return [r for r, d, _ in os.walk(base) if {'art_painting', 'cartoon', 'photo', 'sketch'} <= set(d) and is_rgb(r)]
matches = find_pacs(INPUT)
if not matches:
    for z in glob.glob(f'{INPUT}/**/PACS*.zip', recursive=True):
        zipfile.ZipFile(z).extractall(f'{WORK}/PACS_raw')
    matches = find_pacs(f'{WORK}/PACS_raw')
assert matches, 'No RGB copy of PACS found in the inputs.'
DATA = matches[0]
n_images = sum(len(f) for _, _, f in os.walk(DATA))
print('PACS root:', DATA, '| images:', n_images)
assert n_images == 9991, f'Expected 9,991 images, found {n_images}.'

# Record the environment with the results (the paper reports the platform of every run)
os.makedirs(f'{OUT}/env', exist_ok=True)
stamp = time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())
env = {'platform': 'kaggle', 'gpu': GPU, 'torch': torch.__version__, 'torchvision': torchvision.__version__,
       'cuda': torch.version.cuda, 'python': platform.python_version(), 'started_utc': stamp, 'parts': RUN_PARTS,
       'code_zip_patched_for_truncated_images': PATCHED}
json.dump(env, open(f'{OUT}/env/kaggle_{stamp}.json', 'w'), indent=1)
print(env)

In [ ]:
# 2. From the second session on: copy progress from the previous version's output (added as an input).
# Also reads m2cl_v2 zips or folders downloaded from Colab. Files already here are kept, so rerunning is safe.
PARTS = ('lr', 'round3', 'round3_vlcs', 'env')

def target_for(name):
    parts = name.replace('\\', '/').split('/')
    if not parts[-1] or parts[-1].endswith('.tmp'):
        return None
    for i in range(len(parts) - 2):
        if parts[i].startswith('m2cl_v2') and parts[i + 1] in PARTS:
            return os.path.join(OUT, *parts[i + 1:])
    return None

copied, sources = 0, set()
def copy_in(open_src, name, label):
    global copied
    target = target_for(name)
    if target is None or os.path.exists(target):
        return
    os.makedirs(os.path.dirname(target), exist_ok=True)
    with open_src() as src, open(target + '.tmp', 'wb') as dst:
        shutil.copyfileobj(src, dst)
    os.replace(target + '.tmp', target)
    copied += 1
    sources.add(label)

for root, _, files in os.walk(INPUT):
    for f in files:
        path = os.path.join(root, f)
        if f.startswith('m2cl_v2') and f.endswith('.zip'):
            with zipfile.ZipFile(path) as z:
                for info in z.infolist():
                    if not info.filename.endswith('/'):
                        copy_in(lambda: z.open(info), info.filename, path)
        else:
            copy_in(lambda: open(path, 'rb'), path, path.replace(os.sep, '/').split('/m2cl_v2')[0])
print(f'Copied {copied} files from: {sorted(sources) or "nothing (first session)"}')
if len(sources) > 1:
    print('WARNING: progress came from more than one input. Keep only the newest version of this notebook as an input.')

In [ ]:
# 3. The run plan and where each run stands. Run this any time.
import yaml
import pandas as pd
from m2cl.engine import RunConfig

PACS_T = ('art_painting', 'cartoon', 'photo', 'sketch')
VLCS_T = ('Caltech101', 'LabelMe', 'SUN09', 'VOC2007')
CFG = {m: f'configs/round3_{m}.yaml' for m in ('m2', 'm2cl', 'erm')}
DIR = {('pacs', 'm2'): f'{OUT}/lr/lr_0.001', ('pacs', 'm2cl'): f'{OUT}/round3/m2cl', ('pacs', 'erm'): f'{OUT}/round3/erm',
       ('vlcs', 'm2'): f'{OUT}/round3_vlcs/m2', ('vlcs', 'm2cl'): f'{OUT}/round3_vlcs/m2cl', ('vlcs', 'erm'): f'{OUT}/round3_vlcs/erm'}

def run_dir(method, data, target, seed):
    # The same name train_lodo.py gives the run
    cfg = {k: v for k, v in yaml.safe_load(open(CFG[method])).items() if k not in ('targets', 'seeds')}
    return os.path.join(DIR[(data, method)], RunConfig(target=target, seed=seed, **cfg).run_name)

# Same runs as the Colab notebook. Within a part the fold and seed are the outer loops, so whole pairs finish first.
JOBS = ([('A', m, 'pacs', t, s) for t in PACS_T for s in (0, 1, 2) for m in ('m2', 'm2cl') if not (m == 'm2' and s == 0)]
        + [('B', 'erm', 'pacs', t, s) for t in PACS_T for s in (0, 1, 2)]
        + [('C', m, 'vlcs', t, s) for t in VLCS_T for s in (0, 1, 2) for m in ('m2', 'm2cl', 'erm')]
        + [('D', m, 'pacs', t, s) for t in PACS_T for s in (3, 4) for m in ('m2', 'm2cl')])
assert [sum(j[0] == p for j in JOBS) for p in 'ABCD'] == [20, 12, 36, 16]

def run_status(d):
    result = os.path.join(d, 'result.json')
    if os.path.exists(result) and json.load(open(result)).get('status') == 'complete':
        return 'finished'
    for name in ('latest.pt', 'latest_prev.pt'):
        try:
            return f"continues at epoch {torch.load(os.path.join(d, name), map_location='cpu', weights_only=False)['epoch'] + 1}"
        except Exception:
            pass
    return 'not started'

status = pd.DataFrame([{'part': p, 'run': f'{m} {data} {t} seed {s}', 'status': run_status(run_dir(m, data, t, s))}
                       for p, m, data, t, s in JOBS])
display(status.groupby('part', sort=False).status.apply(lambda s: f"{(s == 'finished').sum()} of {len(s)} finished").to_frame())

## VLCS (only when `RUN_PARTS` includes C)
Uses a VLCS folder from the inputs if one has DomainBed's exact image counts; otherwise streams the Hugging Face copy of DomainBed's VLCS (`PTeterwak/DomainBed_OOP`, `all/VLCS.tar.gz`, 3.8 GB) straight to `/tmp`, so nothing is added to the output. DomainBed's own Google Drive link is skipped: it is rate-limited (docs/AUDIT.md, F15). The cell stops unless each domain has exactly DomainBed's image count.

In [ ]:
EXPECTED = {'Caltech101': 1415, 'LabelMe': 2656, 'SUN09': 3282, 'VOC2007': 3376}   # DomainBed: 10,729 images
IMG_EXTS = ('.jpg', '.jpeg', '.png')                                                 # as in src/m2cl/splits.py
MIRROR = 'https://huggingface.co/datasets/PTeterwak/DomainBed_OOP/resolve/main/all/VLCS.tar.gz'

def image_counts(root):
    return {t: sum(f.lower().endswith(IMG_EXTS) for _, _, fs in os.walk(os.path.join(root, t)) for f in fs)
            for t in VLCS_T}
def find_vlcs(base):
    return [r for r, d, _ in os.walk(base) if set(VLCS_T) <= set(d) and image_counts(r) == EXPECTED]

VLCS = None
if 'C' in RUN_PARTS:
    found = find_vlcs(INPUT) or find_vlcs(f'{WORK}/VLCS_raw')
    if not found:
        print('Streaming VLCS from the Hugging Face mirror (3.8 GB)...')
        shutil.rmtree(f'{WORK}/VLCS_raw', ignore_errors=True)
        os.makedirs(f'{WORK}/VLCS_raw')
        subprocess.run(f'set -o pipefail; wget -q -O - {MIRROR} | tar xz -C {WORK}/VLCS_raw',
                       shell=True, executable='/bin/bash')
        found = find_vlcs(f'{WORK}/VLCS_raw')
    assert found, f'No copy of VLCS with the expected counts {EXPECTED}. Check the internet setting and run again.'
    VLCS = found[0]
    print('VLCS root:', VLCS, '| images per domain:', image_counts(VLCS))
    # Record which images are truncated (they are decoded as far as the file goes, as in DomainBed)
    from PIL import Image, ImageFile
    ImageFile.LOAD_TRUNCATED_IMAGES = False
    truncated = []
    for r, _, fs in os.walk(VLCS):
        for f in fs:
            if f.lower().endswith(IMG_EXTS):
                try:
                    Image.open(os.path.join(r, f)).convert('RGB')
                except OSError as e:
                    truncated.append([os.path.relpath(os.path.join(r, f), VLCS), str(e)])
    ImageFile.LOAD_TRUNCATED_IMAGES = True
    json.dump(truncated, open(f'{OUT}/env/vlcs_truncated_images.json', 'w'), indent=1)
    print(f'Truncated VLCS images: {len(truncated)} (listed in env/vlcs_truncated_images.json)')
else:
    print('Part C is not in RUN_PARTS; VLCS is not needed.')

## Train
Runs one (fold, seed) at a time, in the order of the plan, and skips finished runs. Before each run it checks the clock: it starts a run only if that run can finish before Kaggle's 12-hour limit (judged from the longest run so far, 45 minutes until one finishes), so the session ends cleanly and its output is saved. Never change a setting here: the pre-registration fixes them.

In [ ]:
SESSION_LIMIT_H = 12.0     # Kaggle stops a session after 12 hours
SAFETY_H = 0.3            # setup time before this notebook's clock started, and saving the output

def train(method, data, target, seed):
    root = {'pacs': DATA, 'vlcs': VLCS}[data]
    cmd = [sys.executable, '-u', 'scripts/train_lodo.py', '--config', CFG[method], '--targets', target,
           '--seeds', str(seed), '--data_root', root, '--out_dir', DIR[(data, method)]]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    return p.wait()

longest_h = 0.75
todo = [j for j in JOBS if j[0] in RUN_PARTS and run_status(run_dir(*j[1:])) != 'finished']
print(f'{len(todo)} runs to do in parts {RUN_PARTS}.')
for part, method, data, target, seed in todo:
    elapsed_h = (time.time() - START) / 3600
    if elapsed_h + 1.2 * longest_h + SAFETY_H > SESSION_LIMIT_H:
        print(f'\nStopping after {elapsed_h:.1f} h so the session ends before the 12-hour limit. '
              'Save a new version with this one as an input to continue.')
        break
    print(f'\n=== {part}: {method} {data} {target} seed {seed} (session time {elapsed_h:.1f} h)')
    t0 = time.time()
    code = train(method, data, target, seed)
    assert code == 0, f'The run failed (exit code {code}); read the lines above.'
    longest_h = max(longest_h, (time.time() - t0) / 3600)

display(pd.DataFrame([{'part': p, 'run': f'{m} {data} {t} seed {s}', 'status': run_status(run_dir(m, data, t, s))}
                      for p, m, data, t, s in JOBS if p in RUN_PARTS]).groupby('part', sort=False).status
        .apply(lambda s: f"{(s == 'finished').sum()} of {len(s)} finished").to_frame())

In [ ]:
# Pre-registered analysis of what this platform has finished (the full analysis, with the Colab runs, is done on the laptop)
subprocess.run([sys.executable, 'scripts/analyze_round3.py', OUT, '--json', f'{OUT}/round3_analysis_kaggle.json'])
os.makedirs('/kaggle/working', exist_ok=True)
print('Output size: %.1f GB (Kaggle keeps up to 20 GB)' % (sum(os.path.getsize(os.path.join(r, f))
      for r, _, fs in os.walk('/kaggle/working') for f in fs) / 1e9))

## When you're done
Open the newest version (the notebook's Version history) > Output > download `m2cl_v2` (or Download All). Put it in the laptop's `Downloads` folder and tell Claude. The `env/` files in it record the GPU, torch and CUDA versions of each session, which the paper reports.